In [ ]:
include("../RayTracing.jl")

In [ ]:
parsed_args = RayTracing.parse_commandline()
    
parsed_args["scene-number"] = 1

In [ ]:
# set up logging
logger = RayTracing.setup_logging(parsed_args["debug"])
RayTracing.global_logger(logger)

# set random seed
RayTracing.Random.seed!(parsed_args["seed"])

In [ ]:
I, scene = RayTracing.build_scene(parsed_args)
wbounds = RayTracing.world_bounds(scene.b)

In [ ]:
max_voxels = parsed_args["n-voxels"]

diag = RayTracing.diagonal(wbounds)
bmax = maximum(diag)
n_voxels = max.(1, Int.(round.(diag / bmax * max_voxels)))

In [ ]:
p = RayTracing.Pnt3(0, 0, 0)
offset = RayTracing.offset(wbounds, p)
p_i = clamp.(floor.(offset .* n_voxels), 0, n_voxels .- 1)


p0 = p_i ./ n_voxels
p1 = (p_i .+ 1) ./ n_voxels
voxel_bounds = RayTracing.Bounds3(
    RayTracing.lerp(p0, wbounds.pMin, wbounds.pMax),
    RayTracing.lerp(p1, wbounds.pMin, wbounds.pMax)
)

n_samples = 128
light_contribution = zeros(Float64, length(scene.lights))

In [ ]:
invalidPackedPos = 0xffffffffffffffff
hashTableSize = 4 * n_voxels[1] * n_voxels[2] * n_voxels[3]

# Convert C++ packed position to Julia
packedPos = (UInt64(pi[1]) << 40) | (UInt64(pi[2]) << 20) | UInt64(pi[3])
@assert packedPos != invalidPackedPos

# Compute hash value from packed voxel coordinates
# Following the bit mixing algorithm from:
# http://zimbry.blogspot.ch/2011/09/better-bit-mixing-improving-on.html
hash = packedPos
hash = xor(hash, hash >> 31)
hash *= 0x7fb5d329728ea185
hash = xor(hash, hash >> 27)
hash *= 0x81dadef4bc2dd44d
hash = xor(hash, hash >> 33)
hash = hash % hashTableSize

@assert hash >= 0

hash

In [ ]:
0x000000000002d1fe
0x000000000001d28c